In [1]:
import requests
from io import BytesIO
import rasterio
import xml.etree.ElementTree as ET
from pystac.extensions.table import TableExtension


In [2]:


state = "uttar_pradesh"
district = "jaunpur"
block = "badlapur"

GEOSERVER_WCS_URL = "https://geoserver.core-stack.org:8443/geoserver/LULC_level_1/wcs"
coverage_id = "LULC_level_1:LULC_23_24_badlapur_level_1"

params = {
    "service": "WCS",
    "version": "2.0.1",
    "request": "GetCoverage",
    "CoverageId": coverage_id,
    #"bbox": bbox,
    "format": "geotiff"
}


In [3]:
response = requests.get(GEOSERVER_WCS_URL, params=params, verify=False)
response.raise_for_status()
raster_data = BytesIO(response.content)

with rasterio.open(raster_data) as src:
    bbox = src.bounds
    print(f"Bounding Box: {bbox}")

/home/vishnu/.local/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'geoserver.core-stack.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Bounding Box: BoundingBox(left=82.15758026583272, bottom=25.74454823299613, right=82.64644344345056, top=26.02751754749378)


In [4]:
params = {
    "service": "WCS",
    "version": "2.0.1",
    "request": "GetCoverage",
    "CoverageId": coverage_id,
    "format": "geotiff"
}

try:

    response = requests.get(GEOSERVER_WCS_URL, params=params, verify=False)
    response.raise_for_status()

    raster_data = BytesIO(response.content)

    with rasterio.open(raster_data) as src:
        gsd_x, gsd_y = src.res
        print(f"Ground Sample Distance (GSD):")
        print(f"Pixel Width (x): {gsd_x}")
        print(f"Pixel Height (y): {gsd_y}")
        
        width, height = src.shape
        print(f"Image Dimensions:")
        print(f"  Width: {width} pixels")
        print(f"  Height: {height} pixels")

        if src.crs:
            epsg_code = src.crs.to_string()
            print(f"  CRS: {epsg_code}")
        else:
            print("EPSG Code: Not found or not valid.")
            
except requests.exceptions.RequestException as e:
    print(f"An error occurred while fetching data from GeoServer: {e}")

/home/vishnu/.local/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'geoserver.core-stack.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Ground Sample Distance (GSD):
Pixel Width (x): 8.983152841195215e-05
Pixel Height (y): 8.983152841195215e-05
Image Dimensions:
  Width: 3150 pixels
  Height: 5442 pixels
  CRS: EPSG:4326


### To fetch the data from geoserver for vector layer (admin boundary) and make in tabular format.

In [5]:
import geopandas as gpd
import os
import requests
import pystac
from datetime import datetime, timezone
import pandas as pd
from shapely.geometry import mapping, box
from pystac.extensions.table import TableExtension
vector_desc_df = pd.DataFrame(columns=['column_name', 'column_description'])

In [6]:

geojson_url = "https://geoserver.core-stack.org:8443/geoserver/panchayat_boundaries/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=panchayat_boundaries%3Ajaunpur_badlapur&maxFeatures=50&outputFormat=application%2Fjson"


In [7]:
response = requests.get(geojson_url)
response.raise_for_status()

In [8]:
gdf = gpd.read_file(response.text)
print(gdf.head())


                   id  ADI_2001  ADI_2011  ADI_2019  ASSET_2001  ASSET_2011  \
0  jaunpur_badlapur.1         0         0         0           0           0   
1  jaunpur_badlapur.2         0         0         0           0           0   
2  jaunpur_badlapur.3         0         0         0           0           0   
3  jaunpur_badlapur.4         0         0         0           0           0   
4  jaunpur_badlapur.5         0         0         0           0           0   

   ASSET_2019 BF_2001  BF_2011  BF_2019  ...  TOT_P  block_cen  dist_cen  \
0           0       0        0        0  ...      0          0         0   
1           0       0        0        0  ...      0          0         0   
2           0       0        0        0  ...      0          0         0   
3           0       0        0        0  ...      0          0         0   
4           0       0        0        0  ...      0          0         0   

   district          state  state_cen    tehsil  vill_ID    vill_nam

In [9]:
gdf_wgs84 = gdf.to_crs(epsg=4326) if gdf.crs is None or gdf.crs.to_epsg() != 4326 else gdf

bounds = gdf_wgs84.total_bounds
bbox = [float(b) for b in bounds]
geom = mapping(gdf_wgs84.union_all())

In [10]:
item_id = os.path.splitext(os.path.basename(geojson_url))[0]
item = pystac.Item(
        id=item_id,
        bbox=list(bbox),
        geometry=geom,
        datetime=datetime.now(timezone.utc),
        properties={
            "title": "title"
        }
    )

In [11]:
vector_merged_df = gdf.dtypes.reset_index()
vector_merged_df.columns = ['column_name', 'column_dtype']
vector_merged_df = vector_merged_df.merge(vector_desc_df, on='column_name', how='left').fillna('')




In [12]:
column_data = [
    {
        "name": row['column_name'],
        "type": str(row['column_dtype']),
        "description": row['column_description']
    }
    for _, row in vector_merged_df.iterrows()
]

In [13]:
table_ext = TableExtension.ext(item, add_if_missing=True)
table_ext.columns = column_data


In [14]:
print("Tabular data of the STAC Item's columns:")
tabular_output = pd.DataFrame(column_data)
print(tabular_output)

Tabular data of the STAC Item's columns:
          name      type description
0           id    object            
1     ADI_2001     int32            
2     ADI_2011     int32            
3     ADI_2019     int32            
4   ASSET_2001     int32            
5   ASSET_2011     int32            
6   ASSET_2019     int32            
7      BF_2001    object            
8      BF_2011     int32            
9      BF_2019     int32            
10     FC_2001     int32            
11     FC_2011     int32            
12     FC_2019     int32            
13       F_ILL     int32            
14       F_LIT     int32            
15        F_SC     int32            
16        F_ST     int32            
17    MSW_2001     int32            
18    MSW_2011     int32            
19    MSW_2019     int32            
20       M_ILL     int32            
21       M_LIT     int32            
22        M_SC     int32            
23        M_ST     int32            
24       No_HH     int32          